# Side tests · hand-in-hand checks of the paper's numbers
Read-only on the pipeline: reads stage d, f, g files and writes small summaries to `<work>/side_tests/nb/`.
Run top to bottom on Casper JupyterHub (kernel `npl`). By default it **reads the outputs of `run_side_tests.pbs`** (5 years for Task 1, 10 years for Tasks 3–5), so every cell is quick. Set `RUN = True` to recompute for the `YEARS` you list (about a minute per year and event) into a separate folder.

Weighting: the 1D histograms in Task 2 show cos(lat) and unweighted side by side; everything else is unweighted.

In [ ]:
%matplotlib inline
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
HERE = pathlib.Path.cwd()
REPO = HERE.parent
sys.path[:0] = [str(HERE), str(REPO / "src"), str(REPO / "figures")]
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
from tpl_mll.config import load
import t2_figdata_numbers as T2, t345_events as T345, lcl_compare as L1

RUN = False                       # False: read the PBS outputs · True: recompute YEARS into side_tests/nb/
YEARS = [2000]
OUT = pathlib.Path(load("paths")["work"]) / "side_tests" / ("nb" if RUN else "")
OUT.mkdir(parents=True, exist_ok=True)
EVENTS = ["T", "M"]
show = lambda p: display(Markdown(pathlib.Path(p).read_text()))
print("work:", load("paths")["work"], "\nout:", OUT)

## 0 · Are the files current?
Stage f must carry the stable / diluted-out codes (rerun 2026-10-28) for Task 3. d and e may show *stale* only because `ztop.py` changed; their numbers did not.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "launch", "status", "--stages", "d", "e", "f", "g"],
                     cwd=REPO, capture_output=True, text=True).stdout[-3000:])

## Task 2 · Paper numbers straight from the figure data (all years, no event files)
Exact versions of numbers we have been reading off figures: share of events above zero, tails, means, and the δ_below / δ_above claims.
**Look for:** weighted − unweighted shifts (expect ≤ 0.2 for T, ~0.5–0.9 for M); largest δ_below near the draft's 1.3; mean δ_a for z_LCL < 1 km near 8–9.

In [ ]:
T2.main(["--out", str(OUT)])  # prints the report; also saved as t2_report.md

In [ ]:
# δ_below and δ_above vs z_LCL (unstable events, ε from figdata.yaml)
import figdata as FD
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, e in zip(axes, EVENTS):
    d = FD.FigData(e)
    s, ez = d.hist["lb_zLCL_LPL"], d.edges["zlcl"]
    k = 8                                               # 25 m bins → 200 m
    n = FD.coarsen(s[:, 0, 0], ez, k)[0]
    for q, lab in ((0, "δ_a"), (1, "δ_below"), (2, "δ_above")):
        sm, ec = FD.coarsen(s[:, q, 1], ez, k)
        ax.plot(FD.centers(ec) / 1000, np.where(n >= 200, sm / np.maximum(n, 1), np.nan), label=lab)
    ax.axhline(0, color="k", lw=0.6); ax.set(title=f"{e} · {d.label()}", xlabel="z_LCL of LPL air [km]")
axes[0].set_ylabel("mean [kJ/kg]"); axes[0].legend()

## Task 1 · LCL of LPL air vs LCL of 2 m air
The paper uses the LPL one; this checks the choice does not matter. **Decision rule:** EQUIVALENT if median |dz| < 100 m and no bin with n ≥ 500 moves δ_below or δ_above by ≥ 0.2 kJ/kg.
With `RUN = True`: about 1–3 minutes per year and event (it relifts the plume for every unstable event).

In [ ]:
for y in YEARS if RUN else []:
    for e in EVENTS:
        L1.run(y, e, REPO, OUT / "lcl")
HAVE_LCL = any((OUT / "lcl").glob("lcl_*.npz"))
if HAVE_LCL:
    L1.report(OUT / "lcl", quiet=True)
    show(OUT / "lcl" / "lcl_report.md")
else:
    print("No lcl_*.npz in", OUT / "lcl", "· check the side_tests.log for 'FAILED t1'")

In [ ]:
# dz distribution per event type
fig, ax = plt.subplots(figsize=(6, 3.5))
for e in EVENTS if HAVE_LCL else []:
    h = sum(np.load(p)["h_dz"] for p in (OUT / "lcl").glob(f"lcl_{e}_*.npz"))
    ax.plot(0.5 * (L1.DZ_EDGES[1:] + L1.DZ_EDGES[:-1]), h / h.sum(), label=e)
ax.set(xlim=(-400, 400), xlabel="z_LCL,LPL − z_LCL,2m [m]", ylabel="share per 10 m"); ax.legend()

## Tasks 3–5 · event-level checks
One pass over each year's events collects everything for Tasks 3, 4 and 5 (with `RUN = True`, about a minute per year and event).

In [ ]:
for y in YEARS if RUN else []:
    for e in EVENTS:
        T345.run(y, e, OUT / "t345")
R = {}
for p in sorted((OUT / "t345").glob("t345_*_*.npz")):
    _, e, y = p.stem.split("_")
    R.setdefault(e, {})[int(y)] = dict(np.load(p))
print({e: sorted(v) for e, v in R.items()})

### Task 3 · do the paper's convective events (δ_a > 0) look sensible in stage f?
(a) Stable events with δ_a > 0 should not exist: if h_LPL > h*500 the undiluted plume is buoyant at 500 hPa. Unstable with δ_a ≤ 0 is real (h* lowest away from 500 hPa) and its size says how often 500 hPa is a poor reference.
(b) Of the δ_a > 0 events, how many pass at each ε?

In [ ]:
CAT = FD.STRIKE_CAT
for e, yrs in R.items():
    ys = [y for y, d in yrs.items() if "t3_2x2" in d]
    if not ys:
        print(e, ": no Task 3 data (stage f files lack the new codes, rerun f)"); continue
    t = sum(yrs[y]["t3_2x2"] for y in ys)
    print(f"{e}  {ys}\n            δ_a>0     δ_a≤0\n  unstable {int(t[0,0]):9,} {int(t[0,1]):9,}\n  stable   {int(t[1,0]):9,} {int(t[1,1]):9,}")

In [ ]:
eps = next(d["eps"] for yrs in R.values() for d in yrs.values() if "eps" in d)
fig, axes = plt.subplots(1, len(R), figsize=(5.5 * len(R), 3.8), sharey=True, squeeze=False)
for ax, (e, yrs) in zip(axes[0], R.items()):
    tab = sum(d["t3_by_class"] for d in yrs.values() if "t3_by_class" in d)[0]      # δ_a > 0: (ε, code)
    sh = tab / tab[:, 1:].sum(1, keepdims=True)
    order = FD.STRIKE_ORDER
    ax.stackplot(eps, *[100 * sh[:, c] for c in order], colors=[CAT[c][1] for c in order],
                 labels=[CAT[c][0] for c in order], alpha=0.9)
    ax.set(title=f"{e} · δ_a > 0 events", xlabel="ε [km⁻¹]", xlim=(eps[0], eps[-1]), ylim=(0, 100))
axes[0][0].set_ylabel("share [%]"); h, l = axes[0][0].get_legend_handles_labels(); axes[0][0].legend(h[::-1], l[::-1], fontsize=7)

### Task 4 · η = h*(p_top) − h*500: how wrong is the 500 hPa reference?
Identity: δ_a = E + η, with E the entrainment loss up to the real top. Small η for pass plumes means δ_a really is the entrainment term.
Closure E − (δ_a − η) should be ~0 (numerics only; fail if p95 > 0.5 kJ/kg).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for e, yrs in R.items():
    s = sum(d["t4_sums"] for d in yrs.values() if "t4_sums" in d)
    axes[0].plot(eps, s[:, 0, 1] / np.maximum(s[:, 0, 0], 1), "-o", label=f"{e} all tops")
    axes[0].plot(eps, s[:, 1, 1] / np.maximum(s[:, 1, 0], 1), "--o", label=f"{e} pass")
    hr = sum(d["t4_res"] for d in yrs.values() if "t4_res" in d)
    axes[1].plot(eps, [T345.pct(hr[k], T345.RES_EDGES, 95) for k in range(len(eps))], "-o", label=e)
axes[0].axhline(0, color="k", lw=0.6); axes[0].set(xlabel="ε [km⁻¹]", ylabel="mean η [kJ/kg]"); axes[0].legend(fontsize=7)
axes[1].axhline(0.5, color="r", ls=":"); axes[1].set(xlabel="ε [km⁻¹]", ylabel="p95 |closure residual| [kJ/kg]"); axes[1].legend()

In [ ]:
# h* across 400–600 hPa (the uniformity appendix)
for e, yrs in R.items():
    hu = sum(d["t4_band_mean_minus_500"] for d in yrs.values()); hr = sum(d["t4_band_range"] for d in yrs.values())
    print(f"{e}: band mean − h*500 median {T345.pct(hu, T345.UNI_EDGES, 50):.2f} (p5 {T345.pct(hu, T345.UNI_EDGES, 5):.2f}, "
          f"p95 {T345.pct(hu, T345.UNI_EDGES, 95):.2f}) · range median {T345.pct(hr, T345.RNG_EDGES, 50):.2f}, p95 {T345.pct(hr, T345.RNG_EDGES, 95):.2f} kJ/kg")

### Task 5 · cascade slope Γ_h per box: constant with RH?
Bin-mean h_LPL against bin-mean z_LCL within each (h500, h*500) box, 5 % RH bins. The model assumes one slope and a flat h*500 inside the box.
Γ_h errors (jackknife over years) are in the written report at the end.

In [ ]:
c = 0.5 * (T345.RH5[1:] + T345.RH5[:-1])
for e, yrs in R.items():
    boxes = next(iter(yrs.values()))["t5_boxes"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    for b, name in enumerate(boxes):
        acc = sum(d["t5_acc"][b] for d in yrs.values()); n = acc[:, 0]
        ok = n >= 30
        with np.errstate(invalid="ignore", divide="ignore"):
            axes[0].plot(acc[ok, 2] / n[ok] / 1000, acc[ok, 1] / n[ok], "-o", ms=3, label=name)
            _, loc = T345.slope(acc, 2, (0, 1))
            axes[1].plot(T345.RH5[1:-1], loc, "-o", ms=3, label=name)
            axes[2].plot(c[ok], acc[ok, 4] / n[ok], "-o", ms=3, label=name)
        for win in ((0.3, 0.6), (0.3, 0.7)):
            print(f"{name} RH {win}: Γ_h = {T345.slope(acc, 2, win)[0]:.2f} (LPL LCL), {T345.slope(acc, 3, win)[0]:.2f} (2 m LCL) kJ/kg/km")
    axes[0].set(xlabel="mean z_LCL (LPL) [km]", ylabel="mean h_LPL [kJ/kg]", title=f"{e} cascade per box")
    axes[1].set(xlabel="RH", ylabel="local Γ_h [kJ/kg/km]", title="slope between neighbouring bins")
    axes[2].set(xlabel="RH", ylabel="mean h*500 [kJ/kg]", title="is h*500 flat in the box?")
    for a in axes: a.legend(fontsize=7)

### Task 5b · the slope in the cascade picture: dh/dz of the profile from the LCL to 600 hPa
Task 5 measured the **locus slope** dh_ML/dz_LCL (how the mixed layer and its LCL move together across RH bins; boundary-layer values only). The cascade figure shows the **profile slope**: the composite environment h falling with height above each bin's LCL. The chain-rule model treats them as one Γ_h; they agree only if the mixed-layer tops sit on a shared free-tropospheric profile.
Per box and 5 % RH bin (≥ 200 events, all 40 years of composites): fit dh/dz on the composite levels between the bin's mean LCL and 600 hPa. **Curvature check:** slope of the lower half vs the upper half of that layer; a big gap means the profile is already bending before 600 hPa.
LCL height above sea level ≈ height of the composite at the mean LPL pressure + mean z_LCL (LPL air, above ground) from Task 5; good to ~50 m over the z_s < 0.5 km boxes.

In [ ]:
P_LAYER_TOP = 600.0
rows = {}
for e, yrs in R.items():
    d = FD.FigData(e)
    p = d.levels.astype(float)
    for b, name in enumerate(next(iter(yrs.values()))["t5_boxes"]):
        acc = sum(dd["t5_acc"][b] for dd in yrs.values()); n = acc[:, 0]
        out = []
        for i in range(len(T345.RH5) - 1):
            if n[i] < 200:
                continue
            comp = FD.composite(d, str(name), T345.RH5[i], T345.RH5[i + 1])
            z, h = comp["Z"] / 9.8, comp["MSE"]
            ok = np.isfinite(z) & np.isfinite(h)
            z_lpl = np.interp(np.log(comp["p_LPL"]), np.log(p[ok]), z[ok])
            z_lcl = z_lpl + acc[i, 2] / n[i] / 1000.0                   # km above sea level
            use = ok & (z > z_lcl) & (p >= P_LAYER_TOP)
            if use.sum() < 3:
                continue
            zz, hh = z[use], h[use]
            full = np.polyfit(zz, hh, 1)[0]
            mid = 0.5 * (zz.min() + zz.max())
            lo, hi = zz <= mid, zz >= mid
            low = np.polyfit(zz[lo], hh[lo], 1)[0] if lo.sum() >= 2 else np.nan
            up = np.polyfit(zz[hi], hh[hi], 1)[0] if hi.sum() >= 2 else np.nan
            out.append((0.5 * (T345.RH5[i] + T345.RH5[i + 1]), full, low, up, z_lcl, use.sum()))
        rows[(e, str(name))] = (np.array(out), T345.slope(acc, 2, (0.30, 0.60))[0])

fig, axes = plt.subplots(1, len(rows), figsize=(4.2 * len(rows), 3.8), sharey=True)
for ax, ((e, name), (a, locus)) in zip(np.atleast_1d(axes), rows.items()):
    if len(a):
        ax.plot(a[:, 0], a[:, 1], "k-o", ms=3, label="profile, LCL→600")
        ax.plot(a[:, 0], a[:, 2], "v", color="#1e88e5", ms=4, label="lower half")
        ax.plot(a[:, 0], a[:, 3], "^", color="#e53935", ms=4, label="upper half")
    ax.axhline(locus, color="#8e24aa", ls="--", label=f"locus Γ_h (RH .3–.6) {locus:.1f}")
    ax.axvspan(0.30, 0.60, color="#fff59d", alpha=0.3, zorder=0)
    ax.set(title=name, xlabel="RH", xlim=(0, 1))
np.atleast_1d(axes)[0].set_ylabel("dh/dz [kJ/kg/km]"); np.atleast_1d(axes)[0].legend(fontsize=7)
for (e, name), (a, locus) in rows.items():
    w = a[(a[:, 0] > 0.30) & (a[:, 0] < 0.60)] if len(a) else a
    if len(w):
        print(f"{name}: profile slope in RH .3–.6 mean {w[:, 1].mean():.2f} (range {w[:, 1].min():.2f} to {w[:, 1].max():.2f}), "
              f"lower−upper half {np.nanmean(w[:, 2] - w[:, 3]):+.2f} · locus {locus:.2f} kJ/kg/km")

In [ ]:
# eye check: composite h(z) of one box for a few RH bins, the LCL→600 hPa layer bold
BOX = "M_TPL"                                       # any of the four
e = BOX[0]; d = FD.FigData(e); p = d.levels.astype(float)
fig, ax = plt.subplots(figsize=(5.5, 5))
for lo in (0.30, 0.40, 0.50, 0.60, 0.70):
    comp = FD.composite(d, BOX, lo, lo + 0.05)
    z, h = comp["Z"] / 9.8, comp["MSE"]
    ln, = ax.plot(h, z, lw=1, label=f"RH {lo:.2f}–{lo + .05:.2f}")
    a = rows.get((e, BOX), (np.zeros((0, 6)),))[0]
    k = np.nonzero(np.isclose(a[:, 0], lo + 0.025))[0] if len(a) else []
    if len(k):
        zl = a[k[0], 4]; m = (z > zl) & (p >= P_LAYER_TOP)
        ax.plot(h[m], z[m], lw=3, color=ln.get_color()); ax.plot(np.interp(zl, z[::-1], h[::-1]), zl, "o", color=ln.get_color())
ax.set(ylim=(0, 8), xlabel="composite h [kJ/kg]", ylabel="z [km]", title=f"{BOX}: profiles, LCL (dot) to 600 hPa (bold)")
ax.legend(fontsize=7)

In [ ]:
# full written report for the years run here
T345.report(OUT / "t345"); show(OUT / "t345" / "t345_report.md")